# Resumen Ejecutivo: Análisis de Simulación de Operaciones (Banco de la Nación - Puno)

Este documento presenta los resultados formales derivados de la simulación de eventos discretos correspondientes a la sucursal del Banco de la Nación en Puno. El análisis estadístico y visual tiene como objetivo diagnosticar el comportamiento de la red de cajeros automáticos (ATMs) y evaluar la calidad de servicio brindada a los usuarios, cuantificando tiempos de espera, saturación de la sucursal y confiabilidad técnica del equipamiento.

Los indicadores aquí expuestos sirven como base interpretativa para la toma de decisiones operativas y estratégicas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración visual ejecutiva
sns.set_theme(style="white", context="notebook")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11


## 1. Integración de Datos

A continuación, se procesan los registros generados por la simulación. La información está dividida en tres dimensiones operativas:
*   **Log de Clientes:** Registra la traza temporal de cada usuario desde su llegada hasta su salida (atención exitosa o abandono).
*   **Log de ATMs:** Registra los estados operativos y técnicos de las terminales (operatividad, fallas, mantenimiento y falta de efectivo).
*   **Log del Sistema:** Registra el estado macro del banco minuto a minuto (niveles de congestión).


In [ ]:
data_dir = "./data"

try:
    df_customer = pd.read_csv(os.path.join(data_dir, "customer_event_log.csv"))
    df_system = pd.read_csv(os.path.join(data_dir, "system_snapshot_log.csv"))
    df_atm = pd.read_csv(os.path.join(data_dir, "atm_state_log.csv"))
    
    # Preprocesamiento básico
    df_customer['waiting_time_min'] = df_customer['waiting_time_sec'] / 60
    if 'timestamp' in df_system.columns:
        df_system['timestamp'] = pd.to_datetime(df_system['timestamp'])
        
    print("Datos integrados exitosamente.")
    print(f"Volumen analizado: {len(df_customer):,} interacciones de clientes, {len(df_system):,} snapshots del sistema y {len(df_atm):,} eventos de cajeros.")
except Exception as e:
    print(f"Error al cargar los datos: {e}")


## 2. Indicadores Clave de Rendimiento (KPIs) Operativos

En esta sección evaluamos los indicadores fundamentales del servicio al cliente y el rendimiento global del sistema de colas.


In [ ]:
# Cálculo de KPIs Ejecutivos
total_clientes = len(df_customer)
clientes_atendidos = df_customer['served_flag'].sum()
clientes_abandono = df_customer['abandoned'].sum()
tasa_abandono = (clientes_abandono / total_clientes) * 100

tiempo_espera_promedio = df_customer['waiting_time_min'].mean()
tiempo_espera_p95 = df_customer['waiting_time_min'].quantile(0.95)

longitud_cola_promedio = df_system['queue_length_total'].mean()
longitud_cola_maxima = df_system['queue_length_total'].max()

# Creación de tabla resumen
kpi_data = {
    'Métrica Operativa': [
        'Total de Llegadas de Clientes', 
        'Tasa de Abandono Global', 
        'Tiempo de Espera Promedio', 
        'Tiempo de Espera (Percentil 95)',
        'Longitud de Cola Promedio (Clientes)',
        'Pico Máximo de Cola (Clientes)'
    ],
    'Valor Registrado': [
        f"{total_clientes:,}", 
        f"{tasa_abandono:.2f}%", 
        f"{tiempo_espera_promedio:.1f} minutos", 
        f"{tiempo_espera_p95:.1f} minutos",
        f"{longitud_cola_promedio:.1f}",
        f"{longitud_cola_maxima:,}"
    ]
}

df_kpis = pd.DataFrame(kpi_data)
df_kpis.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
df_kpis


In [ ]:
# Visualización: Distribución de Tiempos de Espera
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]})

sns.histplot(df_customer['waiting_time_min'], bins=50, kde=True, color='#2c3e50', ax=ax1)
ax1.axvline(tiempo_espera_promedio, color='#e74c3c', linestyle='--', linewidth=2, label=f'Promedio: {tiempo_espera_promedio:.1f} min')
ax1.axvline(tiempo_espera_p95, color='#f39c12', linestyle=':', linewidth=2, label=f'Percentil 95: {tiempo_espera_p95:.1f} min')
ax1.set_title('Distribución de Tiempos de Espera en Sucursal', pad=15)
ax1.set_xlabel('Tiempo de Espera (Minutos)')
ax1.set_ylabel('Frecuencia (N° de Clientes)')
ax1.legend(frameon=False)
sns.despine(ax=ax1)

if 'hour_block' in df_customer.columns:
    sns.boxplot(data=df_customer, x='hour_block', y='waiting_time_min', color='#3498db', ax=ax2, fliersize=1)
    ax2.set_title('Fluctuación del Tiempo de Espera por Hora', pad=15)
    ax2.set_xlabel('Hora del Día')
    ax2.set_ylabel('Minutos')
    sns.despine(ax=ax2)

plt.tight_layout()
plt.show()


**Interpretación Estadística:**
La distribución de los tiempos de espera exhibe típicamente una cola larga hacia la derecha. Si el Promedio difiere significativamente de la Mediana, indica que un segmento de clientes experimenta tiempos extremos, reflejado en el **Percentil 95**. Los periodos horarios con mayor varianza en los diagramas de caja (boxplots) denotan las ventanas críticas donde la capacidad de atención es rebasada por la tasa de llegada de clientes.


## 3. Análisis de Deserción y Capacidad de Retención

La deserción o abandono es un síntoma directo de fricción en la experiencia del cliente. Se categoriza estructuralmente para identificar cuellos de botella.


In [ ]:
df_abandono = df_customer[df_customer['abandoned'] == True]

if len(df_abandono) > 0 and 'abandon_reason' in df_abandono.columns:
    razones_abandono = df_abandono['abandon_reason'].value_counts()
    porcentajes = (razones_abandono / razones_abandono.sum()) * 100
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Gráfico de barras horizontales (Estilo Pareto)
    bars = sns.barplot(x=porcentajes.values, y=porcentajes.index, palette='crest', ax=ax)
    
    # Añadir anotaciones
    for i, p in enumerate(bars.patches):
        width = p.get_width()
        ax.text(width + 0.5, p.get_y() + p.get_height()/2. + 0.1, 
                f'{width:.1f}%', ha="left", color='black', fontsize=11)

    ax.set_title('Proporción de Causales de Abandono de Clientes', pad=15)
    ax.set_xlabel('Porcentaje (%) del Total de Abandonos')
    ax.set_ylabel('')
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("No se registraron abandonos significativos en la muestra o la columna no existe.")


**Diagnóstico:**
Un porcentaje alto de abandono debido a *saturación* sugiere una demanda muy por encima de la capacidad máxima teórica de las terminales. Causales relacionadas a *falla percibida* u *horarios* señalan oportunidades de mejora en la disponibilidad técnica y la comunicación de servicio.


## 4. Congestión Sistémica Dinámica

Se examina la serie temporal macro del sistema para identificar periodos sostenidos de estrés operativo y cuantificar la demanda acumulada.


In [ ]:
if 'timestamp' in df_system.columns and 'queue_length_total' in df_system.columns:
    # Remuestrear a intervalos de 15 minutos para suavizar la serie
    df_sys_time = df_system.set_index('timestamp')
    resampled_queue = df_sys_time['queue_length_total'].resample('15T').mean()
    resampled_failed = df_sys_time['failed_atm_count'].resample('15T').mean()
    
    fig, ax1 = plt.subplots(figsize=(16, 6))
    
    # Plot principal: Longitud de cola
    ax1.plot(resampled_queue.index, resampled_queue, color='#2980b9', linewidth=2, label='Promedio Móvil (Cola)')
    ax1.fill_between(resampled_queue.index, resampled_queue, alpha=0.1, color='#2980b9')
    ax1.set_xlabel('Línea de Tiempo Operativa')
    ax1.set_ylabel('Magnitud de la Cola (N° Promedio de Clientes)', color='#2980b9', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#2980b9')
    ax1.set_title('Evolución de la Congestión y Fallas de ATMs (Agrupación: 15 min)', pad=15)
    
    # Eje secundario para fallas
    ax2 = ax1.twinx()
    ax2.plot(resampled_failed.index, resampled_failed, color='#c0392b', linewidth=2, linestyle='-.', label='Terminales Caídas')
    ax2.set_ylabel('N° de ATMs Inoperativos', color='#c0392b', fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='#c0392b')
    ax2.set_ylim(0, df_system['failed_atm_count'].max() + 1)
    
    fig.tight_layout()
    plt.show()


## 5. Auditoría de Confiabilidad de Cajeros Automáticos (ATMs)

La fiabilidad del hardware es la variable crítica. Analizamos el ciclo de vida de los estados del cajero para medir su eficiencia real frente a tiempos muertos (downtime).


In [ ]:
estados = df_atm['atm_state'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de dona para estados de ATM
wedges, texts, autotexts = ax1.pie(estados.values, labels=estados.index, autopct='%1.1f%%', 
                                   startangle=140, colors=sns.color_palette("muted", len(estados)),
                                   wedgeprops=dict(width=0.4, edgecolor='w'))
ax1.set_title('Distribución Global de Estados de Operatividad', pad=15)
plt.setp(autotexts, size=10, weight="bold", color="white")

# Tipos de Fallas
if 'failure_type' in df_atm.columns:
    df_fallas = df_atm[df_atm['failure_flag'] == True]
    if len(df_fallas) > 0:
        conteo_fallas = df_fallas['failure_type'].value_counts()
        sns.barplot(x=conteo_fallas.values, y=conteo_fallas.index, palette='rocket', ax=ax2)
        ax2.set_title('Clasificación de Incidencias Técnicas', pad=15)
        ax2.set_xlabel('Cantidad de Eventos')
        sns.despine(ax=ax2)
    else:
        ax2.text(0.5, 0.5, 'Sin fallas registradas', horizontalalignment='center', verticalalignment='center')
        ax2.axis('off')

plt.tight_layout()
plt.show()


## Conclusiones Estratégicas

El marco de simulación provee evidencia empírica clara sobre el comportamiento de la sucursal de Puno:
1. **Puntos Críticos de Demanda**: (Sustentado en el Análisis de Clientes y Congestión).
2. **Confiabilidad del Hardware**: (Sustentado en el uptime del equipo frente a fallos y cashouts técnicos).
3. **Optimización Sugerida**: Mitigar el factor de abandono mayoritario (congestión vs técnicas) para mejorar sustancialmente el ratio de retención y servicio.
